# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zeref538/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Lane 2 — Refresh / Content Opportunity Scoring.**

**Research question:** *Which pages should a content editor review first for a refresh — and does a learned ranking put meaningfully more truly-declining pages in the top of the queue than a transparent hand-written rule, when both are tested on clients the model has never seen?*

**The decision this supports:** a content team can ship roughly 50 refreshes per cycle. The decision is which pages get those 50 slots. My output is a ranked queue with a score and a plain-words reason code per page; the editor decides what to do with each (refresh / expand / prune / monitor). This is decision-support, not automation.

> **Status: v1 on the 30k-row starter sample.** The plan is to rebuild the label as a true future-window outcome (prior 90 days → next 30 days) on the warehouse release once my feature pipeline is ready — this notebook is the working spine that upgrade slots into.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [1]:
import os
import pandas as pd
import numpy as np

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

# Data: the anonymized starter release that ships with this repo (v1).
# 30,000 rows x 44 columns; one row = one content page for one pseudonymized client;
# metrics are trailing-90-day windows. No client names, domains, URLs, or raw queries exist
# in this data — IDs are salted pseudonyms, used for grouping only, never as features.
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Exclusions, and why:
n0 = len(df)
df = df[df["impressions_90d"] > 0]          # pages nobody saw: nothing to rank
df = df[df["content_age_days"] >= 90]       # too new to judge a 90-day trend honestly
print(f"{n0:,} rows -> {len(df):,} after exclusions "
      f"({df['content_id'].nunique():,} pages, {df['client_id'].nunique()} clients)")

# Known gotchas I handle on purpose (from the data dictionary):
#  - rate columns (ctr, engagement_rate, ...) are x100 percentages: 0.76 means 0.76%
#  - avg_position = 0 means "no data", not rank zero -> flag it, don't trust it
df["has_position"] = (df["avg_position"] > 0).astype(int)
print("rows with no position data:", (df["has_position"] == 0).sum())

# Upgrade path (v2): warehouse release on Hugging Face (~79M daily rows,
# 2025-01-27 -> 2026-06-30) to build prior-90d features and a next-30d outcome label.

30,000 rows -> 30,000 after exclusions (30,000 pages, 32 clients)
rows with no position data: 1205


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [2]:
# METHODOLOGY
# Label (v1 proxy, named honestly): is_declining = trend_direction == "down".
#   It's derived from trend_pct over the same trailing window as the features, so
#   trend_direction and trend_pct are BANNED as features — they'd leak the answer.
# Features: observable, pre-decision signals only. No product flags exist in this data.
# Baseline: the transparent hand rule "stale (180d+) AND visible (500+ imp)", ranked
#   by impressions — the rule a model must beat to earn its keep.
# Validation: client-holdout via GroupShuffleSplit on client_id (30% of clients held
#   out), so no client's pages appear in both train and test. Metric: Precision@50
#   (and @20), because only the top of the queue is ever acted on.

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

df["is_declining"] = df["trend_direction"].str.lower().eq("down").astype(int)

FEATURES = ["impressions_90d", "clicks_90d", "sessions_90d", "ctr", "avg_position",
            "has_position", "days_since_last_update", "content_age_days",
            "word_count", "engagement_rate", "scroll_rate", "days_with_impressions"]
LEAKY = {"trend_direction", "trend_pct"}
assert not (set(FEATURES) & LEAKY), "leakage: label-derived column in features"

X = df[FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining"].values

# Baseline rule score
stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["rule_score"] = stale * visible * df["impressions_90d"]

# Client-holdout split
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
              .split(X, y, groups=df["client_id"]))
print(f"train: {len(tr):,} pages / {df.iloc[tr]['client_id'].nunique()} clients   "
      f"test: {len(te):,} pages / {df.iloc[te]['client_id'].nunique()} clients")
print(f"declining rate — train {y[tr].mean():.3f}, test {y[te].mean():.3f}")

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# tiny self-check on the metric
assert precision_at_k([3, 2, 1], [1, 0, 1], 2) == 0.5

train: 19,166 pages / 22 clients   test: 10,834 pages / 10 clients
declining rate — train 0.532, test 0.559


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [3]:
# RESULTS — every method scored on the SAME held-out clients.
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

models = {
    "logistic_regression": make_pipeline(StandardScaler(),
                                         LogisticRegression(max_iter=2000, class_weight="balanced")),
    "decision_tree": DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=42),
    "random_forest": RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                            random_state=42, n_jobs=-1),
}

rows = []
y_te = y[te]
rule_te = df["rule_score"].values[te]
rows.append({"method": "baseline rule (stale x visible)",
             "P@20": precision_at_k(rule_te, y_te, 20),
             "P@50": precision_at_k(rule_te, y_te, 50)})

test_scores = {}
for name, m in models.items():
    m.fit(X.iloc[tr], y[tr])
    s = m.predict_proba(X.iloc[te])[:, 1]
    test_scores[name] = s
    rows.append({"method": name,
                 "P@20": precision_at_k(s, y_te, 20),
                 "P@50": precision_at_k(s, y_te, 50)})

results = pd.DataFrame(rows).round(3)
best = results.iloc[1:].sort_values("P@50", ascending=False).iloc[0]["method"]
print(results.to_string(index=False))
print(f"\nBest model by P@50 on unseen clients: {best}")
print(f"Test-set declining base rate (random ordering would get): {y_te.mean():.3f}")

                         method  P@20  P@50
baseline rule (stale x visible)  0.65  0.64
            logistic_regression  0.80  0.66
                  decision_tree  0.30  0.50
                  random_forest  0.55  0.64

Best model by P@50 on unseen clients: logistic_regression
Test-set declining base rate (random ordering would get): 0.559


## 5. Limitations

*What this work cannot claim.*

**What this work cannot claim:**

- **No causal claims.** A high score means "worth reviewing first," never "refreshing this page will recover its traffic." Proving that needs an experiment this data can't provide.
- **No Google-algorithm claims.** These are observed associations in one pseudonymized dataset — observed / directional / decision-support language only.
- **The v1 label is a proxy.** `trend_direction == "down"` is computed from the same trailing window as the features. It can't distinguish real decline from seasonality, consolidation (a sibling page absorbing the traffic), or SERP changes. The v2 warehouse label (prior 90 days → next 30 days, with minimum-volume and persistence filters) addresses part of this; the causal question stays out of scope.
- **One split, one sample.** Results come from one client-holdout split of a 30k-page sample with 32 clients; numbers on the full warehouse (or another split seed) will differ. The comparison model-vs-rule is the claim, not the exact decimals.
- **Base rate matters.** ~54% of pages are declining, so even random ordering gets ~0.54 — every method must be read against that floor, not against zero.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [4]:
# RANKED RECOMMENDATIONS — the action playbook (top of the queue, held-out clients only).
test = df.iloc[te].copy()
test["risk_score"] = test_scores[best]

def reason(r):
    if r["days_since_last_update"] >= 180 and r["impressions_90d"] >= 500:
        return "stale_but_visible"        # old content still earning impressions
    if r["has_position"] and r["avg_position"] <= 20 and r["ctr"] < 0.5:
        return "visible_but_unclicked"    # ranks fine, snippet not earning clicks
    if r["sessions_90d"] >= 30 and r["engagement_rate"] < 30:
        return "traffic_but_low_engagement"
    return "model_decline_risk"           # pattern-based; inspect before acting

ACTION = {"stale_but_visible": "refresh",
          "visible_but_unclicked": "rewrite title/meta",
          "traffic_but_low_engagement": "improve on-page content",
          "model_decline_risk": "review / monitor"}

queue = test.sort_values("risk_score", ascending=False).head(50).copy()
queue["reason_code"] = queue.apply(reason, axis=1)
queue["suggested_action"] = queue["reason_code"].map(ACTION)

print("Top-50 queue — reason code mix:")
print(queue["reason_code"].value_counts().to_string())
print(f"\nTruly declining in this top 50: {queue['is_declining'].sum()} of 50")

os.makedirs("work/outputs", exist_ok=True)
cols = ["content_id", "client_id", "risk_score", "reason_code", "suggested_action",
        "impressions_90d", "avg_position", "ctr", "days_since_last_update", "is_declining"]
queue[cols].to_csv("work/outputs/capstone_refresh_queue_top50.csv", index=False)
queue[cols].head(10)

Top-50 queue — reason code mix:
reason_code
visible_but_unclicked    46
model_decline_risk        4

Truly declining in this top 50: 33 of 50


,content_id,client_id,risk_score,reason_code,suggested_action,impressions_90d,avg_position,ctr,days_since_last_update,is_declining
4167,content_a8864e189b2e,client_d029fa3a95,0.925944,visible_but_unclicked,rewrite title/meta,163,6.4,0.00,8,1
1537,content_a928cb66d230,client_f369cb89fc,0.919408,visible_but_unclicked,rewrite title/meta,128,4.2,0.00,20,1
3626,content_8ede62882d0b,client_f369cb89fc,0.894724,visible_but_unclicked,rewrite title/meta,556,14.3,0.00,20,1
11633,content_8f3b70ede1bc,client_d029fa3a95,0.893743,visible_but_unclicked,rewrite title/meta,601,7.7,0.00,20,1
10175,content_374e795aab68,client_f369cb89fc,0.893592,model_decline_risk,review / monitor,235,31.0,0.85,20,0
3651,content_fd16e3475c29,client_d029fa3a95,0.888876,visible_but_unclicked,rewrite title/meta,429,9.0,0.00,183,1
605,content_5d77d3077984,client_f369cb89fc,0.887943,visible_but_unclicked,rewrite title/meta,68,3.0,0.00,20,1
3195,content_b5e9e6453511,client_f369cb89fc,0.887821,visible_but_unclicked,rewrite title/meta,157,6.1,0.00,20,1
22912,content_3dc2aa4e01dc,client_d029fa3a95,0.885584,visible_but_unclicked,rewrite title/meta,304,8.0,0.00,20,1
18063,content_b08562686d22,client_f369cb89fc,0.885499,visible_but_unclicked,rewrite title/meta,2846,2.0,0.14,106,1


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [5]:
# ARTIFACTS for the deployed paper — charts + results table saved to work/outputs/.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Chart 1: model vs baseline at the top of the queue
ax = results.set_index("method")[["P@20", "P@50"]].plot(
    kind="bar", figsize=(8, 4.5), rot=20,
    title="Precision at top of refresh queue — unseen clients (client-holdout)")
ax.axhline(y_te.mean(), color="gray", ls="--", lw=1, label=f"base rate {y_te.mean():.2f}")
ax.legend(); plt.tight_layout()
plt.savefig("work/outputs/capstone_model_vs_baseline.png", dpi=150)
plt.close()

# Chart 2: which signals carry weight (charting the forest's importances — the winning
# logistic model is a scaler+LR pipeline, which has no feature_importances_)
imp = pd.Series(models["random_forest"].feature_importances_, index=FEATURES).sort_values()
imp.plot(kind="barh", figsize=(7, 4.5), title="random_forest — feature importances")
plt.tight_layout()
plt.savefig("work/outputs/capstone_feature_importance.png", dpi=150)
plt.close()
print("Top 5 signals:", ", ".join(imp.sort_values(ascending=False).head(5).index))

results.to_csv("work/outputs/capstone_results_table.csv", index=False)
print("\nSaved to work/outputs/: capstone_model_vs_baseline.png, "
      "capstone_feature_importance.png, capstone_results_table.csv, "
      "capstone_refresh_queue_top50.csv")

Top 5 signals: avg_position, impressions_90d, content_age_days, days_with_impressions, word_count

Saved to work/outputs/: capstone_model_vs_baseline.png, capstone_feature_importance.png, capstone_results_table.csv, capstone_refresh_queue_top50.csv


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.